ChromaDB in python course

[Source code from Swapnil Sharma](https://github.com/sharmasw/Library_experiments/blob/main/chromadb/Chromadb%20Crash%20course.ipynb)

In [ ]:
!pip install chromadb

In [2]:
import chromadb

#Client

In ChromaDB, the client object is your primary interface for interacting with your vector database. It acts as a gateway, allowing you to connect to the database, manage collections of embeddings, and perform operations such as adding, querying, and deleting data. Essentially, it's the main handle through which you communicate your intentions to the ChromaDB system, whether it's a local instance running in-memory or a persistent database on disk.

In [3]:
client = chromadb.Client()

In [4]:
collection = client.create_collection(name="my_collection")

In [5]:
collection.add(
    documents=[
        "This is a document about pineapple. This is a document about oranges",
        " Welcome to Natural Language Processing  It is one of the most exciting research areas as of today  We will see how Python can be used to work with text files. "
    ],
    ids=["id1", "id2"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 9.20MiB/s]


#Query to the Collection

Chroma db will automatically convert our text to embeddings and then with simmilarity factors will find the closest responses to our query.

ChromaDB absolutely stores and processes given prompts, queries, or inputs by converting them into vectors, also known as embeddings.

When you add documents to a collection using collection.add(), ChromaDB takes your raw text and uses an embedding model to transform that text into high-dimensional numerical vectors.

These vectors capture the semantic meaning of the text. Similarly, when you issue a query using collection.query(), your query text is also converted into an embedding.

#Similarity Algorithm

The similarity algorithm primarily used by ChromaDB for comparing these vector embeddings is often cosine similarity. This algorithm measures the cosine of the angle between two vectors in a multi-dimensional space, determining how similar their directions are. A cosine similarity close to 1 indicates high similarity, while a value close to -1 indicates high dissimilarity. Although the specific implementation can vary and other distance metrics like Euclidean distance (L2 distance) can also be used, cosine similarity is a very common choice for semantic similarity searches with embeddings.

In [8]:



query_results = collection.query(
    query_texts=["This is a query document about hawaii"],
    n_results=1
)

print("query results:\n", query_results)



query results:
 {'ids': [['id1']], 'embeddings': None, 'documents': [['This is a document about pineapple. This is a document about oranges']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[1.082511067390442]]}


In [9]:
query_results["documents"][0]

['This is a document about pineapple. This is a document about oranges']

In [10]:
query_results = collection.query(
    query_texts=["Welcome to Image Processing Program"],
    n_results=1
)

print("query results:\n", query_results)


query results:
 {'ids': [['id2']], 'embeddings': None, 'documents': [[' Welcome to Natural Language Processing  It is one of the most exciting research areas as of today  We will see how Python can be used to work with text files. ']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[1.3668313026428223]]}


In [11]:
query_results["documents"][0]

[' Welcome to Natural Language Processing  It is one of the most exciting research areas as of today  We will see how Python can be used to work with text files. ']

#Sotre the data primarly on the "Disk"

In [12]:
client = chromadb.PersistentClient(path="/db/")

Check if Database is Alive?

In [13]:
client.heartbeat()

1765846416752777583

  # Lets be professional 😎

In [14]:
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

In [15]:
DEFAULT_TENANT, DEFAULT_DATABASE, Settings

('default_tenant', 'default_database', chromadb.config.Settings)

In [16]:
client = chromadb.PersistentClient(
    path="/second-db/",
    settings=Settings(
                is_persistent = True, #Now its PersistentClient!
                persist_directory = "/second-db/",
                allow_reset = True,
        anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

In [17]:
client.reset()

True

#Embedding Zone 🤐

In [18]:
from chromadb.utils import embedding_functions

In [19]:
emb_fun=embedding_functions.SentenceTransformerEmbeddingFunction()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
emb_fun.embed_query("Test Query")[0][0]

np.float32(-0.058957532)

Define model:

In [24]:
model_name = "all-MiniLM-L6-v2"

emb_fun = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

In [25]:
len(emb_fun(["TEST QUERY - EMBEDDING STRING"])[0])

384

In [26]:
collection = client.create_collection(
        name="test_collection",
        metadata={"hnsw:space": "cosine"} # l2 is the default
    )

In [27]:
collection.add(
    documents=["som2", "doc2", "doc3","doc4"],
    metadatas=[{"chapter": "3", "verse": "16"}, {"chapter": "3", "verse": "5"}, {"chapter": "29", "verse": "11"},{"chapter": "30", "verse": "12"}],
    ids=["id1", "id2", "id3", "id4"]
)

In [28]:
collection.count()

4

In [35]:
collection.query(
    query_texts="doc4",
    n_results=1,
    where={"chapter": "30"},
)

{'ids': [['id4']],
 'embeddings': None,
 'documents': [['doc4']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'chapter': '30', 'verse': '12'}]],
 'distances': [[1.1920928955078125e-07]]}